In [1]:
import pandas as pd
from transformers import ProphetNetForConditionalGeneration, ProphetNetTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from datasets import Dataset, concatenate_datasets
import evaluate
import utils

<a name='1'></a>
## 1 - Import the Dataset

Bagian ini mempersiapkan dataset yang berasal dari dua sumber yaitu The First Certificate in English (FCE) corpus dan Birkbeck Misspelled Words

* Dataset FCE untuk Grammar Error Correction
* Dataset Birkbeck untuk Misspelled Correction

<a name='6-1'></a>
### 1.1 Grammar Error Dataset

load_dataset_from_file menghasilkan target_text dengan mengubah sebagian kata input_text berdasarkan edits(kata, index) untuk menghasilkan teks dengan grammar benar

In [2]:
train_dataset = utils.load_dataset_from_file(r"dataset/fce/json/fce.train.json")
eval_dataset = utils.load_dataset_from_file(r"dataset/fce/json/fce.dev.json")
test_dataset = utils.load_dataset_from_file(r"dataset/fce/json/fce.test.json")

df = pd.DataFrame(train_dataset,  columns=['input_text', 'target_text'])
print(df.sample(5))

                                             input_text  \
666   Benoni's school is a recognised school for tou...   
137   Hello Kim:\n\nThank you for your letter it was...   
1615  Dear Mrs Ryan\n\nI'm writing concerning your l...   
81    The aim of this report is to suggest which act...   
1391  The respect of Famous people's private life ha...   

                                            target_text  
666   Benoni's school is a recognised school for tou...  
137   Hello Kim:\n\nThank you for your letter. It wa...  
1615  Dear Mrs Ryan\n\nI'm writing concerning your l...  
81    The aim of this report is to suggest which act...  
1391  The  of Famous people's private lives has alwa...  


<a name='6-1'></a>
### 1.2 Misspelled Dataset

make_sentence_pairs dan make_sentence_pairs1 (hanya berbeda kalimat template) menghasilkan kalimat dari kalimat template yang memiliki blank dan diisi oleh misspelled words (input_text) dan correct words (target_text)

In [3]:
file_path = r'dataset/missp.dat.txt'
word_pairs = utils.read_missp_file(file_path)
sentence_pairs = utils.make_sentence_pairs(word_pairs)
sentence_pairs1 = utils.make_sentence_pairs1(word_pairs)

df_train = pd.DataFrame(sentence_pairs, columns=['input_text', 'target_text'])
df_eval = pd.DataFrame(sentence_pairs1, columns=['input_text', 'target_text'])
print(df_train.sample(5))

misspelled_train_dataset = Dataset.from_pandas(df_train)
misspelled_eval_dataset = Dataset.from_pandas(df_eval)

split_ratio = 0.2
num_eval_samples = int(len(misspelled_eval_dataset) * split_ratio)
misspelled_eval_dataset_20 = misspelled_eval_dataset.select(range(num_eval_samples))

                                             input_text  \
31382             I often misspell the word sufficuint.   
15929                     My teacher asked me about go.   
36053                  Please verify if roten is right.   
16637  She wrote handi_craft instead of the right word.   
11241             During lunch, we talked about dieing.   

                                           target_text  
31382            I often misspell the word sufficient.  
15929                  My teacher asked me about good.  
36053                 Please verify if wrote is right.  
16637  She wrote handicraft instead of the right word.  
11241             During lunch, we talked about dying.  


<a name='1'></a>
## 2 - Load ProphetNet pre-trained Model from HuggingFace

In [2]:
model_name = "microsoft/prophetnet-large-uncased"
tokenizer = ProphetNetTokenizer.from_pretrained(model_name)
model = ProphetNetForConditionalGeneration.from_pretrained(model_name)

C:\Users\aurum\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\configuration_utils.py:311: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


<a name='1'></a>
## 3 - Preprocessing the Data

* Bagian ini memproses dataset dengan mengubah pasangan teks input-output (input_text, target_text) menjadi token ID yang bisa diproses oleh ProphetNet untuk sequence-to-sequence learning
* max_target_length di set ke 128 untuk mengimbangi kemampuan GPU dan membuat model lebih terlatih untuk konteks input yang lebih pendek

In [5]:
max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
    model_inputs = tokenizer(examples["input_text"], max_length=max_input_length, truncation=True)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["target_text"], max_length=max_target_length, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

combined_train = concatenate_datasets([train_dataset, misspelled_train_dataset])
combined_eval = concatenate_datasets([eval_dataset, misspelled_eval_dataset_20])
tokenized_train = combined_train.map(preprocess_function, batched=True)
tokenized_eval = combined_eval.map(preprocess_function, batched=True)

Map:   0%|          | 0/38249 [00:00<?, ? examples/s]/home/jupyter-c14220344@john.pet-5348a/nlp-main/prophet/venv/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:3959: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Map: 100%|██████████| 7385/7385 [00:03<00:00, 2459.95 examples/s]


<a name='1'></a>
## 4 - Train the Model

In [6]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./prophetnet-grammar-typo-correction",
    learning_rate=4e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=5,
    predict_with_generate=True,
    logging_dir='./logs',
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [7]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()

/tmp/ipykernel_1767702/3182559150.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss
500,1.304300
1000,0.794600
1500,0.691000
2000,0.632100
2500,0.597900
3000,0.601700
3500,0.554800
4000,0.548100
4500,0.512300
5000,0.477400


/home/jupyter-c14220344@john.pet-5348a/nlp-main/prophet/venv/lib/python3.10/site-packages/transformers/modeling_utils.py:3465: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=47815, training_loss=0.22017075387950125, metrics={'train_runtime': 17076.9864, 'train_samples_per_second': 11.199, 'train_steps_per_second': 2.8, 'total_flos': 2.316446428408627e+16, 'train_loss': 0.22017075387950125, 'epoch': 5.0})

<a name='1'></a>
## 5 - Evaluate the Model using BLEU and ROUGE

In [3]:
model_dir = r"D:\Natural Language Processing\Project_Akhir\ProphetNet\checkpoint-47815"

tokenizer = ProphetNetTokenizer.from_pretrained(model_dir)
model = ProphetNetForConditionalGeneration.from_pretrained(model_dir)

In [4]:
def correct_grammar(sentence: str, max_len: int = 128) -> str:
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    outputs = model.generate(
        inputs["input_ids"],
        max_length=max_len,
        num_beams=5,
        early_stopping=True
    )
    corrected_sentence = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return corrected_sentence

In [ ]:
with open("source.txt", "r", encoding="utf-8") as f:
    data = [line.strip() for line in f]

with open("pred.txt", "w", encoding="utf-8") as pred:
    
    for entry in data:
        pred.write(correct_grammar(entry).replace("\n", " ") + "\n")

In [5]:
with open("pred.txt", "r", encoding="utf-8") as f:
    predictions = [line.strip() for line in f]

with open("target.txt", "r", encoding="utf-8") as f:
    targets = [line.strip() for line in f]

assert len(predictions) == len(targets), "Jumlah prediksi dan referensi harus sama!"

references = [[ref] for ref in targets]

bleu = evaluate.load("bleu")
bleu_score = bleu.compute(predictions=predictions, references=references)
print(f"\nBLEU Score: {bleu_score['bleu']:.4f}")

rouge = evaluate.load("rouge")
rouge_score = rouge.compute(predictions=predictions, references=targets)
print("\nROUGE Scores:")
for key, value in rouge_score.items():
    print(f"{key}: {value:.4f}")


BLEU Score: 0.2845

ROUGE Scores:
rouge1: 0.7398
rouge2: 0.6554
rougeL: 0.7187
rougeLsum: 0.7191


<a name='1'></a>
## 6 - Try to Correct some Sentences!

In [9]:
file_path = r"D:\Natural Language Processing\Project_Akhir\dataset\input.txt"

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

paragraphs = [p.strip() for p in text.strip().split("\n\n") if p.strip()]

for idx, para in enumerate(paragraphs, 1):
    correction = correct_grammar(para)
    print(f"\nParagraf {idx}:")
    print("Input:")
    print(utils.wrap_text_by_words(para))
    print("\nKoreksi:")
    print(utils.wrap_text_by_words(correction))
    print("-" * 80)


Paragraf 1:
Input:
she have many friends and teacher.

Koreksi:
she have many friends and teachers.
--------------------------------------------------------------------------------

Paragraf 2:
Input:
he is a senior docter.

Koreksi:
he is a senior doctor.
--------------------------------------------------------------------------------

Paragraf 3:
Input:
Many student thinks that to learn a forein language is difficult because they
haven't enough oportunity to practise. In the school, they usualy studies
grammar, but not speak much. Also, teacher doesn’t give advices how to improve
listening skills, which make more harder to understand native speakers. Some
have tryed to watch films without subtitel, but it not helped them much.

Koreksi:
many students think that to learn a foreign language is difficult because they
haven't enough opportunities to practise. in school, they usually study grammar,
but not speak much. also, the teacher doesn't give advice about how to improve
listening s